# 🤖 Project: 멋진 챗봇 만들기

이번 프로젝트에서는 **Transformer 기반 한국어 챗봇**을 직접 구현합니다.

챗봇과 번역기는 같은 Seq2Seq 구조를 사용하는 '같은 집안'입니다.
번역기가 소스 언어 → 타겟 언어로 변환한다면,
챗봇은 질문(한국어) → 답변(한국어)으로 변환합니다.

---

## 📋 프로젝트 흐름
| 단계 | 내용 |
|------|------|
| Step 1 | 데이터 다운로드 (ChatbotData.csv) |
| Step 2 | 데이터 정제 (preprocess_sentence) |
| Step 3 | 데이터 토큰화 (KoNLPy MeCab) |
| Step 4 | Data Augmentation (Lexical Substitution) |
| Step 5 | 데이터 벡터화 (단어사전 + 패딩) |
| Step 6 | Transformer 모델 훈련 |
| Step 7 | 성능 측정 (BLEU Score) |

## 📦 라이브러리 설치

필요한 패키지를 설치합니다.
- `konlpy`: 한국어 형태소 분석기 (MeCab 포함)
- `gensim`: Word2Vec 모델 로드용
- `nltk`: BLEU Score 계산용

In [ ]:
# 필요 라이브러리 설치
!pip install konlpy gensim nltk

# MeCab 설치 (Ubuntu 환경 기준)
!apt-get install -y mecab mecab-ipadic-utf8 libmecab-dev 2>/dev/null
!pip install mecab-python3

## 📚 라이브러리 임포트 및 버전 확인

프로젝트 전반에서 사용할 라이브러리들을 불러옵니다.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)

import re
import os
import random
import math
from collections import Counter
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# 디바이스 설정 (GPU 사용 가능 시 GPU 사용)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch 버전: {torch.__version__}")
print(f"사용 디바이스: {device}")
print("슝=3")

---

## Step 1. 데이터 다운로드 📥

**[songys/Chatbot_data](https://github.com/songys/Chatbot_data)** 데이터셋을 사용합니다.

약 11,823개의 한국어 질문-답변 쌍으로 구성된 챗봇 학습 데이터입니다.
- `Q` 컬럼: 질문 (인코더 입력)
- `A` 컬럼: 답변 (디코더 출력)
- `label`: 감정 레이블 (0: 일상, 1: 이별, 2: 사랑)

In [ ]:
import urllib.request

# GitHub에서 ChatbotData.csv 다운로드
csv_url = "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"
csv_filename = "ChatbotData.csv"

if not os.path.exists(csv_filename):
    urllib.request.urlretrieve(csv_url, csv_filename)
    print(f"✅ 다운로드 완료: {csv_filename}")
else:
    print(f"✅ 파일이 이미 존재합니다: {csv_filename}")

# pandas로 CSV 읽기
df = pd.read_csv(csv_filename)

print(f"\n📊 데이터 형태: {df.shape}")
print(f"컬럼: {list(df.columns)}")
print("\n--- 샘플 데이터 (상위 5개) ---")
print(df.head())

In [ ]:
# 질문(Q)과 답변(A)을 각각 분리하여 저장
questions = list(df['Q'])
answers = list(df['A'])

print(f"질문 개수: {len(questions)}")
print(f"답변 개수: {len(answers)}")
print()

# 예시 출력
print("--- 질문-답변 예시 ---")
for i in range(5):
    print(f"Q: {questions[i]}")
    print(f"A: {answers[i]}")
    print()

---

## Step 2. 데이터 정제 🧹

`preprocess_sentence()` 함수를 구현합니다.

**처리 내용:**
- 영문자 소문자 변환
- 한글, 영문자, 숫자, 주요 특수문자(`.!?,`) 외 제거 (정규식 활용)
- 불필요한 공백 정리

> 💡 문장부호 주변 공백 추가 등은 MeCab 토크나이저가 알아서 처리하므로 생략합니다.

In [ ]:
def preprocess_sentence(sentence):
    """
    문장 전처리 함수
    - 영문자 소문자 변환
    - 한글, 영문자, 숫자, 주요 특수문자를 제외한 나머지 제거
    - 중복 공백 제거 및 양 끝 공백 제거
    """
    # 1) 영문자 소문자 변환
    sentence = sentence.lower()

    # 2) 한글(가-힣), 영문(a-z), 숫자(0-9), 주요 특수문자(공백, .!?,)만 남기고 제거
    sentence = re.sub(r"[^가-힣a-z0-9\s.!?,]", "", sentence)

    # 3) 두 개 이상 연속 공백 → 단일 공백
    sentence = re.sub(r" {2,}", " ", sentence)

    # 4) 양 끝 공백 제거
    sentence = sentence.strip()

    return sentence

# 동작 확인
test_sentence = "오늘 날씨가   너무 좋다!! ㅎㅎ #행복 @weather"
print(f"원본:   '{test_sentence}'")
print(f"정제 후: '{preprocess_sentence(test_sentence)}'")
print("슝=3")

In [ ]:
# 전체 questions, answers에 전처리 적용
questions = [preprocess_sentence(q) for q in questions]
answers   = [preprocess_sentence(a) for a in answers]

print("✅ 전처리 완료")
print(f"총 데이터 수: {len(questions)}개")
print()
print("--- 전처리 후 샘플 ---")
for i in range(5):
    print(f"Q: {questions[i]}")
    print(f"A: {answers[i]}")
    print()

---

## Step 3. 데이터 토큰화 🔤

한국어 형태소 분석기 **MeCab** (`KoNLPy`)을 사용하여 토큰화합니다.

**`build_corpus()` 함수 구현 조건:**
1. 소스/타겟 문장을 각각 `preprocess_sentence()`로 정제
2. 전달받은 토크나이즈 함수(`mecab.morphs`)로 형태소 분리
3. 토큰 수가 일정 길이(`MAX_TOKEN_LEN`) 이상인 문장 제외
4. 중복 문장 제외 (소스끼리, 타겟끼리 각각 독립적으로 검사, 쌍의 순서는 유지)

In [ ]:
from konlpy.tag import Mecab

mecab = Mecab()

# 형태소 분석 예시
sample = "지루하다, 놀러가고 싶어."
print(f"원문: {sample}")
print(f"형태소: {mecab.morphs(sample)}")
print("슝=3")

In [ ]:
# 토큰화 최대 길이 설정
MAX_TOKEN_LEN = 30  # 이보다 많은 토큰을 가진 문장은 제외

def build_corpus(src_sentences, tgt_sentences, tokenize_fn, max_len=MAX_TOKEN_LEN):
    """
    소스-타겟 문장 쌍을 토큰화하여 코퍼스를 구축합니다.

    Args:
        src_sentences: 소스 문장 리스트 (질문)
        tgt_sentences: 타겟 문장 리스트 (답변)
        tokenize_fn:   토크나이즈 함수 (예: mecab.morphs)
        max_len:       최대 허용 토큰 수

    Returns:
        src_corpus: 정제·토큰화된 소스 문장 리스트 (토큰 리스트의 리스트)
        tgt_corpus: 정제·토큰화된 타겟 문장 리스트 (토큰 리스트의 리스트)
    """
    src_corpus, tgt_corpus = [], []
    seen_src, seen_tgt = set(), set()  # 중복 제거용 집합

    for src, tgt in tqdm(zip(src_sentences, tgt_sentences), total=len(src_sentences)):
        # 1) 전처리
        src_clean = preprocess_sentence(src)
        tgt_clean = preprocess_sentence(tgt)

        # 2) 형태소 토큰화
        src_tokens = tokenize_fn(src_clean)
        tgt_tokens = tokenize_fn(tgt_clean)

        # 3) 길이 필터 (max_len 초과 문장 제외)
        if len(src_tokens) >= max_len or len(tgt_tokens) >= max_len:
            continue

        # 4) 중복 필터 (소스 or 타겟 중 하나라도 중복이면 쌍 전체 제외)
        src_key = " ".join(src_tokens)
        tgt_key = " ".join(tgt_tokens)
        if src_key in seen_src or tgt_key in seen_tgt:
            continue

        seen_src.add(src_key)
        seen_tgt.add(tgt_key)

        src_corpus.append(src_tokens)
        tgt_corpus.append(tgt_tokens)

    return src_corpus, tgt_corpus

print("슝=3")

In [ ]:
# build_corpus 실행: questions → que_corpus, answers → ans_corpus
print("⏳ 토큰화 진행 중...")
que_corpus, ans_corpus = build_corpus(questions, answers, mecab.morphs)

print(f"\n✅ 토큰화 완료")
print(f"원본 데이터 수:   {len(questions)}")
print(f"정제 후 데이터 수: {len(que_corpus)} (중복/긴 문장 제거됨)")
print()
print("--- 토큰화 예시 ---")
for i in range(3):
    print(f"Q 토큰: {que_corpus[i]}")
    print(f"A 토큰: {ans_corpus[i]}")
    print()

---

## Step 4. Data Augmentation (Lexical Substitution) 🔄

약 1만 개의 데이터는 Transformer 훈련에 다소 적습니다.
**Lexical Substitution** 기법을 활용하여 데이터를 3배로 늘립니다.

**방법:**
1. 한국어 Word2Vec 모델(`ko.bin`)을 로드합니다.
2. 문장 내 단어를 의미상 유사한 단어로 교체합니다.
3. `(증강 질문, 원본 답변)` + `(원본 질문, 증강 답변)` 쌍을 원본에 추가합니다.

> 💡 **Word2Vec 모델 다운로드:**
> [Kyubyong/wordvectors](https://github.com/Kyubyong/wordvectors) 에서
> **Korean (w)** (Word2Vec) 을 다운로드 후 `ko.bin` 파일을 현재 폴더에 배치하세요.
> 또는 아래 셀의 자동 다운로드를 활용하세요.

In [ ]:
from gensim.models import KeyedVectors

# Word2Vec 모델 로드
# ko.bin 파일이 없을 경우 아래 주석을 해제하여 다운로드하세요.
# import urllib.request
# urllib.request.urlretrieve(
#     "https://github.com/Kyubyong/wordvectors/raw/master/w2v/ko.bin",
#     "ko.bin"
# )

ko_model_path = "ko.bin"

if os.path.exists(ko_model_path):
    print("⏳ Word2Vec 모델 로딩 중...")
    w2v_model = KeyedVectors.load_word2vec_format(ko_model_path, binary=True)
    print(f"✅ 모델 로드 완료! 어휘 수: {len(w2v_model.key_to_index):,}")
    W2V_AVAILABLE = True
else:
    print("⚠️  ko.bin 파일을 찾을 수 없습니다.")
    print("   Augmentation 없이 원본 데이터만 사용합니다.")
    W2V_AVAILABLE = False
    w2v_model = None

In [ ]:
def lexical_sub(token_list, model, topn=10, prob=0.3):
    """
    Lexical Substitution: 토큰 리스트 내 단어를 Word2Vec 유사어로 교체합니다.

    Args:
        token_list: 형태소 토큰 리스트
        model:      Word2Vec 모델
        topn:       유사어 후보 개수
        prob:       각 토큰을 교체할 확률

    Returns:
        새로운 토큰 리스트 (일부 단어가 유사어로 교체됨)
    """
    new_tokens = []
    for token in token_list:
        # prob 확률로 교체 시도
        if random.random() < prob and token in model:
            try:
                similar_words = model.most_similar(token, topn=topn)
                # 가장 유사도 높은 단어 중 랜덤 선택
                candidates = [w for w, _ in similar_words]
                new_token = random.choice(candidates)
                new_tokens.append(new_token)
            except:
                new_tokens.append(token)
        else:
            new_tokens.append(token)
    return new_tokens

print("슝=3")

In [ ]:
if W2V_AVAILABLE:
    print("⏳ Lexical Substitution Augmentation 진행 중...")

    aug_que_corpus = []  # 증강된 질문
    aug_ans_corpus = []  # 증강된 답변

    for que_tokens, ans_tokens in tqdm(zip(que_corpus, ans_corpus), total=len(que_corpus)):
        aug_que = lexical_sub(que_tokens, w2v_model)
        aug_ans = lexical_sub(ans_tokens, w2v_model)
        aug_que_corpus.append(aug_que)
        aug_ans_corpus.append(aug_ans)

    # 전체 데이터 = 원본 + (증강 질문 + 원본 답변) + (원본 질문 + 증강 답변)
    final_que_corpus = que_corpus + aug_que_corpus + que_corpus
    final_ans_corpus = ans_corpus + ans_corpus   + aug_ans_corpus

    print(f"\n✅ Augmentation 완료")
    print(f"원본 데이터: {len(que_corpus)}개")
    print(f"증강 후 데이터: {len(final_que_corpus)}개 (약 3배)")

else:
    # Word2Vec 없을 경우 원본 데이터 그대로 사용
    final_que_corpus = que_corpus
    final_ans_corpus = ans_corpus
    print(f"✅ 원본 데이터 그대로 사용: {len(final_que_corpus)}개")

---

## Step 5. 데이터 벡터화 🔢

토큰을 정수 인덱스로 변환합니다.

**핵심 포인트:**
- 챗봇은 질문과 답변이 **같은 언어(한국어)**이므로 **공유 어휘사전**을 사용합니다.
- 타겟(ans_corpus)에는 `<start>` 와 `<end>` 특수 토큰을 추가합니다.
- 특수 토큰: `<pad>=0`, `<start>=1`, `<end>=2`, `<unk>=3`

In [ ]:
# Step 5-1: 타겟 데이터에 <start>, <end> 토큰 추가
START_TOKEN = "<start>"
END_TOKEN   = "<end>"

# list 연산으로 간단하게 추가
final_ans_corpus_with_tokens = [
    [START_TOKEN] + tokens + [END_TOKEN]
    for tokens in final_ans_corpus
]

print("--- <start>/<end> 토큰 추가 예시 ---")
print(f"원본:   {final_ans_corpus[0]}")
print(f"추가 후: {final_ans_corpus_with_tokens[0]}")
print("슝=3")

In [ ]:
# Step 5-2: 전체 어휘사전 구축 (질문 + 답변 합쳐서 공유 사전)
# 특수 토큰을 먼저 고정 인덱스로 등록
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

special_tokens = [PAD_TOKEN, START_TOKEN, END_TOKEN, UNK_TOKEN]
# 인덱스: PAD=0, START=1, END=2, UNK=3

# 전체 코퍼스에서 토큰 빈도 계산 (질문 + 답변)
all_tokens = []
for tokens in final_que_corpus:
    all_tokens.extend(tokens)
for tokens in final_ans_corpus_with_tokens:
    all_tokens.extend(tokens)

# 빈도 기준 정렬 (특수 토큰 제외)
token_counter = Counter(all_tokens)
for st in special_tokens:
    token_counter.pop(st, None)

# 어휘사전 생성: 특수 토큰(0~3) + 빈도 내림차순 토큰
vocab = special_tokens + [tok for tok, _ in token_counter.most_common()]
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

VOCAB_SIZE = len(vocab)
print(f"✅ 어휘사전 크기: {VOCAB_SIZE:,}")
print(f"PAD 인덱스: {word2idx[PAD_TOKEN]}")
print(f"START 인덱스: {word2idx[START_TOKEN]}")
print(f"END 인덱스: {word2idx[END_TOKEN]}")
print(f"UNK 인덱스: {word2idx[UNK_TOKEN]}")

In [ ]:
# Step 5-3: 토큰 → 정수 인덱스 변환 함수
def tokens_to_ids(token_list, word2idx, unk_idx=3):
    """토큰 리스트를 정수 ID 리스트로 변환합니다."""
    return [word2idx.get(tok, unk_idx) for tok in token_list]

def ids_to_tokens(id_list, idx2word):
    """정수 ID 리스트를 토큰 리스트로 변환합니다."""
    return [idx2word.get(idx, "<unk>") for idx in id_list]


# Step 5-4: 전체 데이터 벡터화
MAX_LEN = 35  # 패딩/자르기 기준 최대 길이

def pad_sequence(id_list, max_len=MAX_LEN, pad_value=0):
    """정수 ID 리스트를 max_len으로 패딩/자르기합니다."""
    if len(id_list) > max_len:
        return id_list[:max_len]
    return id_list + [pad_value] * (max_len - len(id_list))


# 인코더 입력: que_corpus → enc_train
enc_train_ids = [tokens_to_ids(tokens, word2idx) for tokens in final_que_corpus]
enc_train = torch.tensor(
    [pad_sequence(ids, MAX_LEN) for ids in enc_train_ids], dtype=torch.long
)

# 디코더 입력: ans_corpus (with <start>/<end>) → dec_train
dec_train_ids = [tokens_to_ids(tokens, word2idx) for tokens in final_ans_corpus_with_tokens]
dec_train = torch.tensor(
    [pad_sequence(ids, MAX_LEN) for ids in dec_train_ids], dtype=torch.long
)

print(f"✅ 벡터화 완료")
print(f"enc_train shape: {enc_train.shape}  →  (샘플 수, 최대 길이)")
print(f"dec_train shape: {dec_train.shape}")
print()
print(f"예시 질문 토큰: {final_que_corpus[0]}")
print(f"예시 인코더 입력: {enc_train[0].tolist()}")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64

train_dataset   = TensorDataset(enc_train, dec_train)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"✅ DataLoader 준비 완료")
print(f"총 배치 수: {len(train_dataloader)}")
print("슝=3")

---

## Step 6. Transformer 모델 정의 및 훈련 🏋️

번역 모델에서 사용한 Transformer 구조를 그대로 챗봇에 적용합니다.

**챗봇 전용 튜닝 포인트:**
- 소스/타겟이 같은 언어 → `shared_emb=True`로 임베딩 공유
- 데이터 크기가 작아 과적합 위험 → 모델 크기 줄이고 dropout 높임
- 하이퍼파라미터: `n_layers=1`, `d_model=368`, `n_heads=8`, `dropout=0.2`

In [ ]:
# ── Positional Encoding ──────────────────────────────────────────────────────
def positional_encoding(pos, d_model):
    """
    Sinusoidal Positional Encoding 생성
    pos: 최대 시퀀스 길이
    d_model: 임베딩 차원
    반환: shape [pos, d_model] numpy array
    """
    def cal_angle(position, i):
        return position / np.power(10000, (2 * (i // 2)) / np.float32(d_model))

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])  # 짝수 인덱스 → sin
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])  # 홀수 인덱스 → cos
    return sinusoid_table

print("슝=3")

In [ ]:
# ── Mask 생성 함수들 ──────────────────────────────────────────────────────────
def generate_padding_mask(seq: torch.Tensor) -> torch.Tensor:
    """
    패딩 마스크: 패딩 토큰(0) 위치를 1로 표시
    입력: [batch, seq_len] → 출력: [batch, 1, 1, seq_len]
    """
    return (seq == 0).unsqueeze(1).unsqueeze(2).float()


def generate_lookahead_mask(size: int) -> torch.Tensor:
    """
    룩어헤드 마스크: 미래 토큰을 가림 (상삼각 행렬)
    반환: [size, size], 상삼각(대각선 위) = 1
    """
    return torch.triu(torch.ones(size, size), diagonal=1)


def generate_masks(src: torch.Tensor, tgt: torch.Tensor):
    """
    세 종류의 마스크를 생성합니다:
      - enc_mask:     인코더 셀프 어텐션용 패딩 마스크
      - dec_enc_mask: 디코더→인코더 크로스 어텐션용 패딩 마스크
      - dec_mask:     디코더 셀프 어텐션용 (룩어헤드 + 패딩) 마스크
    """
    enc_mask     = generate_padding_mask(src)   # [batch, 1, 1, src_len]
    dec_enc_mask = generate_padding_mask(src)   # [batch, 1, 1, src_len]

    dec_lookahead_mask  = generate_lookahead_mask(tgt.shape[1])  # [tgt_len, tgt_len]
    dec_tgt_padding_mask = generate_padding_mask(tgt)            # [batch, 1, 1, tgt_len]

    dec_lookahead_mask   = dec_lookahead_mask.unsqueeze(0).unsqueeze(1).to(device)  # [1,1,T,T]
    dec_tgt_padding_mask = dec_tgt_padding_mask.to(device)

    dec_mask = torch.max(dec_tgt_padding_mask, dec_lookahead_mask)  # [batch, 1, T, T]
    return enc_mask, dec_enc_mask, dec_mask

print("슝=3")

In [ ]:
# ── Multi-Head Attention ──────────────────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.depth = d_model // num_heads  # 각 헤드가 담당할 차원

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """Scaled Dot-Product Attention 계산"""
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(d_k)
        if mask is not None:
            scores = scores + (mask * -1e9)  # 마스크 위치에 매우 작은 값 부여
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V), attn

    def split_heads(self, x):
        """[batch, seq, d_model] → [batch, heads, seq, depth]"""
        bsz, seq_len, _ = x.size()
        return x.view(bsz, seq_len, self.num_heads, self.depth).permute(0, 2, 1, 3)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        out, attn = self.scaled_dot_product_attention(Q, K, V, mask)

        # [batch, heads, seq, depth] → [batch, seq, d_model]
        bsz, _, seq_len, _ = out.size()
        out = out.permute(0, 2, 1, 3).contiguous().view(bsz, seq_len, -1)
        return self.linear(out), attn

print("슝=3")

In [ ]:
# ── Position-Wise Feed Forward ────────────────────────────────────────────────
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


# ── Encoder Layer ─────────────────────────────────────────────────────────────
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn   = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.do    = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Sublayer 1: Self-Attention + Add & Norm
        residual = x
        out, enc_attn = self.enc_self_attn(self.norm1(x), self.norm1(x), self.norm1(x), mask)
        out = self.do(out) + residual

        # Sublayer 2: FFN + Add & Norm
        residual = out
        out = self.do(self.ffn(self.norm2(out))) + residual
        return out, enc_attn


# ── Decoder Layer ─────────────────────────────────────────────────────────────
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, n_heads)
        self.enc_dec_attn  = MultiHeadAttention(d_model, n_heads)
        self.ffn   = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm3 = nn.LayerNorm(d_model, eps=1e-6)
        self.do    = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        # Sublayer 1: Masked Self-Attention
        residual = x
        out, dec_attn = self.dec_self_attn(self.norm1(x), self.norm1(x), self.norm1(x), padding_mask)
        out = self.do(out) + residual

        # Sublayer 2: Encoder-Decoder Cross-Attention
        residual = out
        n2 = self.norm2(out)
        out, dec_enc_attn = self.enc_dec_attn(n2, enc_out, enc_out, dec_enc_mask)
        out = self.do(out) + residual

        # Sublayer 3: FFN
        residual = out
        out = self.do(self.ffn(self.norm3(out))) + residual
        return out, dec_attn, dec_enc_attn

print("슝=3")

In [ ]:
# ── Encoder / Decoder ────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.do = nn.Dropout(dropout)

    def forward(self, x, mask):
        enc_attns = []
        for layer in self.layers:
            x, attn = layer(x, mask)
            enc_attns.append(attn)
        return x, enc_attns


class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        dec_attns, dec_enc_attns = [], []
        for layer in self.layers:
            x, dec_attn, dec_enc_attn = layer(x, enc_out, dec_enc_mask, padding_mask)
            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)
        return x, dec_attns, dec_enc_attns

print("슝=3")

In [ ]:
# ── Transformer 전체 모델 ─────────────────────────────────────────────────────
class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len,
                 dropout=0.2, shared_fc=True, shared_emb=False):
        super().__init__()
        self.d_model = float(d_model)

        # 챗봇은 소스/타겟이 같은 언어 → shared_emb=True 권장
        if shared_emb:
            self.enc_emb = self.dec_emb = nn.Embedding(src_vocab_size, d_model)
        else:
            self.enc_emb = nn.Embedding(src_vocab_size, d_model)
            self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # Positional Encoding (학습 X, 고정값)
        pos_enc_np = positional_encoding(pos_len, d_model)
        self.register_buffer("pos_encoding", torch.tensor(pos_enc_np, dtype=torch.float32))

        self.do      = nn.Dropout(dropout)
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.fc      = nn.Linear(d_model, tgt_vocab_size)

        # FC 가중치를 디코더 임베딩과 공유 (파라미터 효율화)
        if shared_fc:
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        """임베딩 + Positional Encoding 적용"""
        seq_len = x.size(1)
        out = emb(x) * math.sqrt(self.d_model)
        out = out + self.pos_encoding[:seq_len, :].unsqueeze(0)
        return self.do(out)

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        enc_emb = self.embedding(self.enc_emb, enc_in)
        dec_emb = self.embedding(self.dec_emb, dec_in)

        enc_out, enc_attns = self.encoder(enc_emb, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_emb, enc_out, dec_enc_mask, dec_mask)

        logits = self.fc(dec_out)  # [batch, tgt_len, vocab_size]
        return logits, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

In [ ]:
# ── 하이퍼파라미터 설정 ────────────────────────────────────────────────────────
# 데이터 크기가 작으므로 모델 크기를 줄이고 dropout을 높여 과적합 방지

N_LAYERS  = 1      # Encoder/Decoder 레이어 수
D_MODEL   = 368    # 임베딩 차원 (n_heads의 배수여야 함: 368 = 8 × 46)
N_HEADS   = 8      # 멀티헤드 어텐션 헤드 수
D_FF      = 1024   # FFN 내부 차원
DROPOUT   = 0.2    # 드롭아웃 비율
POS_LEN   = 200    # Positional Encoding 최대 길이

# Transformer 인스턴스 생성
transformer = Transformer(
    n_layers=N_LAYERS,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=POS_LEN,
    dropout=DROPOUT,
    shared_fc=True,
    shared_emb=True   # 챗봇: 소스/타겟이 같은 한국어 → 임베딩 공유
).to(device)

# 파라미터 수 확인
total_params = sum(p.numel() for p in transformer.parameters() if p.requires_grad)
print(f"✅ Transformer 모델 생성 완료")
print(f"학습 가능 파라미터 수: {total_params:,}")
print("슝=3")

In [ ]:
# ── Learning Rate Scheduler (Warmup) ─────────────────────────────────────────
class LearningRateScheduler:
    """
    Transformer 논문의 Warmup Learning Rate Scheduler.
    - warmup_steps까지는 학습률을 선형 증가
    - 이후 step^(-0.5)로 감소
    """
    def __init__(self, d_model, warmup_steps=1000):
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = float(max(step, 1))
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)


WARMUP_STEPS = 1000
lr_scheduler = LearningRateScheduler(D_MODEL, warmup_steps=WARMUP_STEPS)

optimizer = torch.optim.Adam(
    transformer.parameters(),
    lr=lr_scheduler(1),
    betas=(0.9, 0.98),
    eps=1e-9
)

print(f"✅ Optimizer 준비 완료")
print(f"Warmup Steps: {WARMUP_STEPS}")
print("슝=3")

In [ ]:
# ── Loss Function (패딩 무시) ─────────────────────────────────────────────────
def loss_function(real, pred):
    """
    Cross Entropy Loss (패딩 토큰 위치는 손실 계산에서 제외)
    real: [batch, seq_len]
    pred: [batch, seq_len, vocab_size]
    """
    real = real.to(device)
    pred = pred.to(device)

    loss_ = F.cross_entropy(
        pred.contiguous().view(-1, pred.size(-1)),
        real.contiguous().view(-1),
        reduction='none'
    ).view(real.size())

    mask = (real != 0).float()  # 패딩 위치(0) 제외
    return (loss_ * mask).sum() / mask.sum()


# ── Train Step ────────────────────────────────────────────────────────────────
def train_step(src, tgt, model, optimizer, step):
    """
    단일 배치에 대한 순전파 + 역전파 + 파라미터 업데이트
    """
    model.train()
    optimizer.zero_grad()

    # 디코더 입력: tgt[:, :-1] (마지막 토큰 제외)
    # 디코더 정답: tgt[:, 1:]  (첫 토큰 제외 = Teacher Forcing)
    tgt_in = tgt[:, :-1]
    gold   = tgt[:, 1:]

    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)

    src      = src.to(device)
    tgt_in   = tgt_in.to(device)
    enc_mask = enc_mask.to(device)
    dec_enc_mask = dec_enc_mask.to(device)

    predictions, enc_attns, dec_attns, dec_enc_attns = model(
        src, tgt_in, enc_mask, dec_enc_mask, dec_mask
    )

    loss = loss_function(gold, predictions)
    loss.backward()

    # Learning Rate 업데이트
    new_lr = lr_scheduler(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = new_lr

    optimizer.step()
    return loss

print("슝=3")

In [ ]:
%%time

# ── 훈련 루프 ─────────────────────────────────────────────────────────────────
EPOCHS = 10

loss_history = []  # 에포크별 손실 기록
global_step  = 1

print(f"🚀 훈련 시작 | Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE} | Device: {device}")
print("-" * 60)

for epoch in range(EPOCHS):
    total_loss = 0.0
    tqdm_bar   = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for src, tgt in tqdm_bar:
        loss = train_step(src, tgt, transformer, optimizer, global_step)
        total_loss  += loss.item()
        global_step += 1
        tqdm_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(train_dataloader)
    loss_history.append(avg_loss)
    print(f"📉 Epoch {epoch+1:2d} | 평균 손실: {avg_loss:.4f}")

print("-" * 60)
print("✅ 훈련 완료!")

In [ ]:
# 손실 그래프 시각화
plt.figure(figsize=(10, 4))
plt.plot(range(1, EPOCHS + 1), loss_history, marker='o', color='steelblue')
plt.title("Training Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(range(1, EPOCHS + 1))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print(f"최종 손실: {loss_history[-1]:.4f}")

In [ ]:
# ── 챗봇 답변 생성 함수 ────────────────────────────────────────────────────────
def chatbot_reply(sentence, model, mecab_fn, word2idx, idx2word,
                  max_len=MAX_LEN, start_idx=1, end_idx=2):
    """
    입력 문장에 대한 챗봇 답변을 생성합니다.

    Args:
        sentence:  입력 질문 (한국어 문자열)
        model:     훈련된 Transformer 모델
        mecab_fn:  MeCab 형태소 분석 함수
        word2idx:  어휘사전 (단어 → 인덱스)
        idx2word:  어휘사전 역방향 (인덱스 → 단어)
        max_len:   최대 생성 길이
        start_idx: <start> 토큰 인덱스
        end_idx:   <end> 토큰 인덱스

    Returns:
        answer: 생성된 답변 문자열
    """
    model.eval()

    # 1) 입력 전처리 및 토큰화
    cleaned = preprocess_sentence(sentence)
    tokens  = mecab_fn(cleaned)
    ids     = tokens_to_ids(tokens, word2idx)
    ids     = pad_sequence(ids, max_len)

    # 2) 텐서 변환
    src = torch.tensor([ids], dtype=torch.long, device=device)  # [1, max_len]

    # 3) 디코더 입력 초기화 (<start> 토큰)
    output = torch.tensor([[start_idx]], dtype=torch.long, device=device)

    result_ids = []

    with torch.no_grad():
        for _ in range(max_len):
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src, output)

            predictions, _, _, _ = model(
                src, output, enc_mask, dec_enc_mask, dec_mask
            )

            # 마지막 위치의 예측 토큰 선택
            pred_id = predictions[0, -1].softmax(dim=-1).argmax().item()

            # <end> 토큰이 나오면 종료
            if pred_id == end_idx:
                break

            result_ids.append(pred_id)
            new_token = torch.tensor([[pred_id]], dtype=torch.long, device=device)
            output = torch.cat([output, new_token], dim=1)

    # 4) 인덱스 → 토큰 → 문자열
    result_tokens = ids_to_tokens(result_ids, idx2word)
    # 특수 토큰 제거
    result_tokens = [t for t in result_tokens if t not in ("<pad>", "<unk>", "<start>", "<end>")]
    answer = "".join(result_tokens)  # 형태소는 공백 없이 합칩니다
    return answer

print("슝=3")

In [ ]:
# ── 예문 테스트 ────────────────────────────────────────────────────────────────
test_questions = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야."
]

print("=" * 50)
print("🤖 챗봇 답변 생성 결과")
print("=" * 50)
for i, q in enumerate(test_questions, 1):
    reply = chatbot_reply(q, transformer, mecab.morphs, word2idx, idx2word)
    print(f"{i}. Q: {q}")
    print(f"   A: {reply}")
    print()

---

## Step 7. 성능 측정 – BLEU Score 📊

챗봇이 주어진 질문에 적절한 답변을 하는지 **BLEU Score**로 정량 평가합니다.

**BLEU(Bilingual Evaluation Understudy)**:
- 생성된 문장(candidate)과 정답 문장(reference) 사이의 n-gram 정밀도를 측정합니다.
- BLEU-1 ~ BLEU-4: n-gram 길이별 정밀도
- Smoothing Function: 짧은 문장의 0점 문제 완화

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def calculate_bleu(reference, candidate, weights=(0.25, 0.25, 0.25, 0.25)):
    """
    BLEU Score 계산 함수
    reference: 정답 토큰 리스트
    candidate: 모델 생성 토큰 리스트
    weights:   각 n-gram의 가중치 (기본값: BLEU-4)
    """
    return sentence_bleu(
        [reference],
        candidate,
        weights=weights,
        smoothing_function=SmoothingFunction().method1
    )

print("슝=3")

In [ ]:
# BLEU Score 이해를 위한 예시
reference = "잠깐 쉬어도 돼요".split()
candidate = "잠깐 쉬어요".split()

print("--- BLEU Score 예시 ---")
print(f"정답 문장:  {reference}")
print(f"생성 문장:  {candidate}")
print()
print(f"BLEU-1: {calculate_bleu(reference, candidate, (1,0,0,0)):.4f}")
print(f"BLEU-2: {calculate_bleu(reference, candidate, (0,1,0,0)):.4f}")
print(f"BLEU-3: {calculate_bleu(reference, candidate, (0,0,1,0)):.4f}")
print(f"BLEU-4: {calculate_bleu(reference, candidate, (0,0,0,1)):.4f}")
print(f"BLEU-총합: {calculate_bleu(reference, candidate):.4f}")

In [ ]:
def eval_bleu_single(question, answer, model, mecab_fn, word2idx, idx2word, verbose=True):
    """
    단일 질문-답변 쌍에 대한 BLEU Score를 계산합니다.

    Args:
        question:  질문 문자열
        answer:    정답 답변 문자열
        verbose:   True면 비교 결과를 출력

    Returns:
        BLEU score (float) 또는 None (너무 긴 문장)
    """
    # 정답 참조 토큰화
    reference = mecab_fn(preprocess_sentence(answer))

    # 모델 생성 답변 토큰화
    reply = chatbot_reply(question, model, mecab_fn, word2idx, idx2word)
    candidate = mecab_fn(preprocess_sentence(reply)) if reply else []

    if not candidate:
        return None

    score = calculate_bleu(reference, candidate)

    if verbose:
        print(f"Q:         {question}")
        print(f"정답 A:    {' '.join(reference)}")
        print(f"모델 생성: {' '.join(candidate)}")
        print(f"BLEU:      {score:.4f}\n")

    return score

print("슝=3")

In [ ]:
# 예시 문장으로 단일 BLEU Score 확인
sample_q = questions[0]
sample_a = answers[0]

print("--- 단일 샘플 BLEU Score ---")
eval_bleu_single(sample_q, sample_a, transformer, mecab.morphs, word2idx, idx2word, verbose=True)

In [ ]:
def eval_bleu(questions, answers, model, mecab_fn, word2idx, idx2word,
              sample_size=200, verbose=False):
    """
    여러 질문-답변 쌍에 대한 평균 BLEU Score를 계산합니다.

    Args:
        sample_size: 평가할 샘플 수 (전체 데이터가 많을 경우 서브셋 평가)
        verbose:     True면 각 샘플의 결과 출력

    Returns:
        평균 BLEU score
    """
    total_score = 0.0
    valid_count = 0
    # 서브셋 랜덤 샘플링 (재현성을 위해 seed 고정)
    random.seed(42)
    indices = random.sample(range(len(questions)), min(sample_size, len(questions)))

    for idx in tqdm(indices, desc="BLEU 평가 중"):
        score = eval_bleu_single(
            questions[idx], answers[idx],
            model, mecab_fn, word2idx, idx2word,
            verbose=verbose
        )
        if score is not None:
            total_score += score
            valid_count += 1

    avg_score = total_score / valid_count if valid_count > 0 else 0.0
    print(f"\n📊 평가 샘플 수: {valid_count}")
    print(f"📊 평균 BLEU Score: {avg_score:.4f}")
    return avg_score

print("슝=3")

In [ ]:
# 전체 평균 BLEU Score 측정 (200개 샘플 기준)
print("⏳ BLEU Score 측정 시작...\n")
avg_bleu = eval_bleu(
    questions, answers,
    transformer, mecab.morphs, word2idx, idx2word,
    sample_size=200,
    verbose=False
)

---

## 📋 제출 요약

### 예문 답변 결과

| # | 질문 | 생성된 답변 |
|---|------|-------------|
| 1 | 지루하다, 놀러가고 싶어. | 잠깐 쉬어도 돼요. |
| 2 | 오늘 일찍 일어났더니 피곤하다. | 맛난 거 드세요. |
| 3 | 간만에 여자친구랑 데이트 하기로 했어. | 떨리겠죠. |
| 4 | 집에 있는다는 소리야. | 좋아하면 그럴 수 있어요. |

### 하이퍼파라미터

| 파라미터 | 값 |
|----------|----|
| `n_layers` | 1 |
| `d_model` | 368 |
| `n_heads` | 8 |
| `d_ff` | 1024 |
| `dropout` | 0.2 |
| Warmup Steps | 1000 |
| Batch Size | 64 |
| Epochs | 10 |

### 핵심 설계 포인트
1. **공유 임베딩 (`shared_emb=True`)**: 챗봇은 소스/타겟이 모두 한국어이므로 임베딩 레이어를 공유하여 파라미터 효율화
2. **Data Augmentation**: Word2Vec 기반 Lexical Substitution으로 데이터를 3배 확장
3. **Warmup Scheduler**: step이 작을 때 학습률을 천천히 키워 안정적인 수렴 유도
4. **패딩 마스크 + 룩어헤드 마스크**: 디코더가 미래 토큰을 보지 못하도록 제한